In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
# Initializing Document Loaders
cardiomegaly_loader = PyPDFLoader("data/Cardiomegaly.pdf")
pleuraleffusion_pneumothorax_loader = PyPDFLoader("data/Pleural Effusion & Pneumothorax.pdf")
tuberculosis_loader = PyPDFLoader("data/Tuberculosis_Guidelines.pdf")
xraydictionary_loader = PyPDFLoader("data/XRay_Dictionary.pdf")

In [9]:
# Data Ingestion
import asyncio

# Loading Documents Asynchronously (Concurrent)
cardiomegaly_docs, pleuraleffusion_pneumothorax_docs, tuberculosis_docs, xraydictionary_docs = await asyncio.gather(
    cardiomegaly_loader.aload(),
    pleuraleffusion_pneumothorax_loader.aload(),
    tuberculosis_loader.aload(),
    xraydictionary_loader.aload()
)

In [10]:
# Text Splitting
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

cardiomegaly_docs = text_splitter.split_documents(cardiomegaly_docs)
pleuraleffusion_pneumothorax_docs = text_splitter.split_documents(pleuraleffusion_pneumothorax_docs)
tuberculosis_docs = text_splitter.split_documents(tuberculosis_docs)
xraydictionary_docs = text_splitter.split_documents(xraydictionary_docs)

In [11]:
# Embedding Model (HuggingFace's Cloud Embedding Model)
from langchain_huggingface import HuggingFaceEndpointEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()

embeddings = HuggingFaceEndpointEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=os.getenv("HF_TOKEN")
)

In [12]:
# Vector Store using FAISS
from langchain_community.vectorstores import FAISS
import os

os.makedirs("./faiss_db", exist_ok=True)

cardiomegaly_vector_collection = FAISS.from_documents(
    documents=cardiomegaly_docs,
    embedding=embeddings
)
cardiomegaly_vector_collection.save_local("./faiss_db/cardiomegaly")

pleuraleffusion_pneumothorax_vector_collection = FAISS.from_documents(
    documents=pleuraleffusion_pneumothorax_docs,
    embedding=embeddings
)
pleuraleffusion_pneumothorax_vector_collection.save_local("./faiss_db/pleuraleffusion_pneumothorax")

tuberculosis_vector_collection = FAISS.from_documents(
    documents=tuberculosis_docs,
    embedding=embeddings
)
tuberculosis_vector_collection.save_local("./faiss_db/tuberculosis")

xraydictionary_vector_collection = FAISS.from_documents(
    documents=xraydictionary_docs,
    embedding=embeddings
)
xraydictionary_vector_collection.save_local("./faiss_db/xraydictionary")

In [13]:
# Loading Vector Store For Similarity Search Testing
from langchain_community.vectorstores import FAISS

# Loading the FAISS indexes
cardiomegaly_vector_collection = FAISS.load_local(
    "./faiss_db/cardiomegaly",
    embeddings,
    allow_dangerous_deserialization=True
)

pleuraleffusion_pneumothorax_vector_collection = FAISS.load_local(
    "./faiss_db/pleuraleffusion_pneumothorax",
    embeddings,
    allow_dangerous_deserialization=True
)

tuberculosis_vector_collection = FAISS.load_local(
    "./faiss_db/tuberculosis",
    embeddings,
    allow_dangerous_deserialization=True
)

xraydictionary_vector_collection = FAISS.load_local(
    "./faiss_db/xraydictionary",
    embeddings,
    allow_dangerous_deserialization=True
)

# Test query
results = xraydictionary_vector_collection.similarity_search(
    "What is cardiomegaly ?",
    k=3
)
print(results[0].page_content)

main categories: first, a vascular category (ie, heart, superior 
vena cava, ascending thoracic aorta, aortic arch, descend-
ing thoracic aorta, intrapericardial pulmonary arteries, tho-
racic duct), and second, the trachea, carina, and esophagus, 
which share an embryological origin (the endoderm), as 
well as lymph nodes.
Boundaries of the paravertebral (posterior) compartment 
are superiorly, the thoracic inlet; inferiorly, the diaphragm; 
anteriorly, the visceral compartment; and posterolaterally, 
a vertical line along the posterior margin of the chest wall 
at the lateral aspect of the transverse processes. The major 
contents of this compartment are the thoracic spine and 
paravertebral soft tissues (96,97).
Micronodule (Pathology) (Figs 2 1, 22, 87)
A micronodule is a circumscribed and typically round opac-
ity*, previously defined as less than 3 mm, but currently de-
fined as less than 6 mm in average diameter. This size thresh-


# Some More Cloud Embedding Techniques

In [14]:
# Embedding Model (Google's Cloud Embedding Model)
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(
    api_key = os.getenv("GEMINI_API_KEY"),
    model="models/gemini-embedding-001"
)
vector = embeddings.embed_query("hello, world!")
vector[:5]

ModuleNotFoundError: No module named 'langchain_google_genai'

In [ ]:
# Embedding Model (Cohere's Cloud Embedding Model)
from langchain_cohere import CohereEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()

embeddings = CohereEmbeddings(
    cohere_api_key=os.getenv("COHERE_API_KEY"),
    model="embed-english-light-v3.0"
)
vector = embeddings.embed_query("hello, world!")
vector[:5]